In [1]:
import os
import re
import sys
import pickle
import pandas as pd
import spacy
from tqdm import tqdm

import nltk

Load previously calculated lexicons and annotations

In [2]:
path_annotations = "../data/annotations"
path_lexicons = "../data/lexicons"
path_predictions = "../data/predictions"

os.makedirs(path_predictions, exist_ok=True)

Load the training data

In [3]:
df_train = pickle.load(open(os.path.join(path_annotations, "df_train.pkl"), "rb"))
print(f"Loaded {len(df_train)} annotations")

Loaded 9239 annotations


In [4]:
df_train

,doc_index,doc_id,result_id,word_idx,start,end,label,text,line_number
0,0,19026587,ent0,68,68,69,NEG,no,19026587_0
1,0,19026587,ent1,69,69,72,NSCO,habitos toxicos.,19026587_0
2,0,19026587,ent2,38,38,39,NSCO,cistoscopia,19026587_2
3,0,19026587,ent3,41,41,42,NEG,negativa,19026587_2
4,0,19026587,ent4,42,42,45,NSCO,para lesiones malignas,19026587_2
...,...,...,...,...,...,...,...,...,...
9234,253,20339886,ent40,25,25,26,NSCO,streptotest,20339886_29
9235,253,20339886,ent41,49,49,50,NSCO,urocultivo,20339886_29
9236,253,20339886,ent42,185,185,186,NSCO,hemocultivo,20339886_29
9237,253,20339886,ent43,209,209,210,NSCO,foco,20339886_29


In [5]:
from nltk.tokenize import word_tokenize

In [6]:
import nltk
from typing import List, Dict

# Assume 'grammar' is the nltk.CFG object created successfully in the previous steps
# If not, you need to run the code that builds the lemma_grammar_dict,
# converts it to cfg_string, and then calls nltk.CFG.fromstring(cfg_string)

# --- Simple Parsing (Mapping) Function ---

def parse_tokens_with_lexical_grammar(grammar: nltk.CFG, sentence_tokens: list[str]) -> list[str]:
    """
    Applies a purely lexical NLTK grammar to map tokens in a sentence.

    This function iterates through the grammar's lexical rules (LEMMA -> 'word')
    to build a word-to-lemma lookup map. It then iterates through the input
    tokens, replacing any known token (case-insensitive) with its corresponding
    lemma symbol from the grammar. Tokens not found in the grammar are
    returned unchanged.

    This is NOT syntactic parsing but rather a form of lexical normalization or tagging.

    Args:
        grammar: An nltk.CFG object containing primarily lexical rules.
                 It must have been successfully created (not None).
        sentence_tokens: A list of strings representing the tokenized sentence.

    Returns:
        A list of strings where known tokens are replaced by their lemma
        symbols from the grammar. Returns the original list if grammar is
        invalid or sentence is empty.
    """
    if not isinstance(grammar, nltk.CFG) or not sentence_tokens:
        return sentence_tokens # Return original if grammar invalid or no tokens

    word_to_lemma_map: dict[str, str] = {}
    try:
        for production in grammar.productions():
            # Check if it's a lexical rule like: LEMMA -> 'word'
            if production.is_lexical() and isinstance(production.rhs()[0], str):
                word = production.rhs()[0] # The terminal word from CFG (should be lowercase)
                lemma = production.lhs().symbol() # The non-terminal lemma string (e.g., '_negativo')
                word_to_lemma_map[word] = lemma
    except Exception as e:
        print(f"Error processing grammar productions: {e}")
        return sentence_tokens # Return original tokens on error

    if not word_to_lemma_map:
        print("Warning: No lexical rules found in the grammar to build a map.")
        # Fall through to map_tokens_to_lemmas, which will just return original tokens

    # 2. Map input tokens using the created map
    mapped_output = []
    for token in sentence_tokens:
        # Lookup the lowercase version of the token in the map
        # If not found, default to the original token itself
        lemma = word_to_lemma_map.get(token.lower(), token)
        mapped_output.append(lemma)

    return mapped_output

In [7]:
def preprocess_text(text):
    """
    Preprocess the text for rule-based analysis
    """
    if not text or pd.isna(text):
        return ""
        
    text = text.lower().strip()
    
    return text.strip()


def tokenize_text(text):
    """Split text into tokens (words, punctuation)"""
    if not text:
        return []
    # Simple tokenization
    return word_tokenize(text)


def find_cues(text, cue_lexicon):
    """
    Find all instances of cues in the text
    
    Parameters:
        text list[str]: Tokenized text to search for cues
        cue_lexicon (DataFrame): Lexicon containing cues
        
    Returns:
        list: List of dictionaries with cue information
    """
    if not text:
        return []

    cues = []
    
    lemmas = set(cue_lexicon["lemma"].tolist())

    # Find each cue term in the text
    for term in lemmas:
        indices = []
        indices = [i for i, x in enumerate(text) if x == term]
        for index in indices:
            start = index
            end = index + 1
            cues.append({
                "start": start,
                "end": end,
                "text": " ".join(text[start:end]),
                "token": text[start],
                "lemma": term
            })
    
    # Sort cues by position in text
    cues = sorted(cues, key=lambda x: x["start"])
    return cues



def determine_scope(text, cue):
    """
    Determine the scope of a cue
    
    Parameters:
        text (str): Full text
        cue (dict): Cue information from find_cues
        
    Returns:
        dict: Scope information
    """
    cue_start = cue["start"]
    cue_end = cue["end"]
    
    # TODO Improve scope detection

    # Determine scope direction (forward for most, backward for "sin" ...)
    backward_cues = ["sin", "sense", "excepto", "excepte", "salvo", "tret"]
    direction = "backward" if cue["token"].lower() in backward_cues else "forward"
    
    if direction == "forward":
        # Scope starts after the cue
        scope_start = cue_end
        scope_end = len(text)
        
        # Find the next punctuation or end of text
        for i in range(scope_start, len(text)):
            if text[i] in ".,;:!?":
                scope_end = i
                break
    else:
        # Backward scope ends at the cue
        scope_end = cue_start
        scope_start = 0
        
        # Find the previous punctuation or start of text
        for i in range(scope_end-1, -1, -1):
            if i < 0 or text[i] in ".,;:!?":
                scope_start = i + 1
                break
    
    # Return scope
    return {
        "start": scope_start,
        "end": scope_end,
        "text": text[scope_start:scope_end]
    }



**Document Processing Function**

The `process_document()` function generates predictions for a single document:

For each line of text in the document:
- Preprocess the text (lowercase, extract relevant section)
- Find negation cues by matching terms from the lexicon
- For each negation cue:
   - Create a NEG prediction
   - Determine the scope affected by this negation
   - Create an NSCO prediction for the scope
- Find uncertainty cues by matching terms from the lexicon
- For each uncertainty cue:
   - Create a UNC prediction
   - Determine the scope affected by this uncertainty
   - Create a USCO prediction for the scope

This approach creates predictions based only on text pattern matching, without using existing labels

In [8]:
# load lemmatizer
import pickle
import os
import nltk

path_lexicons = "../data/lexicons"

lemma_grammar_dict = pickle.load(open(os.path.join(path_lexicons, "lemma_grammar_dict.pkl"), "rb"))

def add_lemma(grammar_dict: dict[str, set[str]], input_word: str, target_lemma: str):
    """Adds lowercase input word mapping."""
    if not target_lemma or not input_word: return # Skip empty
    # Store input word in lowercase for case-insensitive lookup later
    grammar_dict[target_lemma].add(input_word.lower())

def convert_dict_to_cfg_string(grammar_dict: dict[str, set[str]]) -> str:
    """Converts dict to CFG string."""
    cfg_rules = []
    for target_lemma in sorted(grammar_dict.keys()):
        # Input words are already lowercase in the set
        input_words = sorted(list(grammar_dict[target_lemma]))
        # Use repr() for proper quoting in CFG string format
        productions = " | ".join(repr(word) for word in input_words)
        # Ensure target_lemma is valid (basic check)
        safe_target = target_lemma.replace('<','').replace('>','') # Use content if <> included
        if not safe_target.replace('_','').isalnum() or not safe_target[0].isalpha():
             print(f"Warning: Target '{target_lemma}' converted to '{safe_target}' may still be invalid Nonterminal.")
        cfg_rules.append(f"{safe_target} -> {productions}")
    return "\n".join(cfg_rules)



# Convert the dictionary to a CFG string
cfg_string = convert_dict_to_cfg_string(lemma_grammar_dict)

grammar = nltk.CFG.fromstring(cfg_string)

In [9]:
import nltk
from typing import List, Dict

# Assume 'grammar' is the nltk.CFG object created successfully in the previous steps
# If not, you need to run the code that builds the lemma_grammar_dict,
# converts it to cfg_string, and then calls nltk.CFG.fromstring(cfg_string)

# --- Simple Parsing (Mapping) Function ---

def parse_tokens_with_lexical_grammar(grammar: nltk.CFG, sentence_tokens: list[str]) -> list[str]:
    """
    Applies a purely lexical NLTK grammar to map tokens in a sentence.

    This function iterates through the grammar's lexical rules (LEMMA -> 'word')
    to build a word-to-lemma lookup map. It then iterates through the input
    tokens, replacing any known token (case-insensitive) with its corresponding
    lemma symbol from the grammar. Tokens not found in the grammar are
    returned unchanged.

    This is NOT syntactic parsing but rather a form of lexical normalization or tagging.

    Args:
        grammar: An nltk.CFG object containing primarily lexical rules.
                 It must have been successfully created (not None).
        sentence_tokens: A list of strings representing the tokenized sentence.

    Returns:
        A list of strings where known tokens are replaced by their lemma
        symbols from the grammar. Returns the original list if grammar is
        invalid or sentence is empty.
    """
    if not isinstance(grammar, nltk.CFG) or not sentence_tokens:
        return sentence_tokens # Return original if grammar invalid or no tokens

    word_to_lemma_map: dict[str, str] = {}
    try:
        for production in grammar.productions():
            # Check if it's a lexical rule like: LEMMA -> 'word'
            if production.is_lexical() and isinstance(production.rhs()[0], str):
                word = production.rhs()[0] # The terminal word from CFG (should be lowercase)
                lemma = production.lhs().symbol() # The non-terminal lemma string (e.g., '_negativo')
                word_to_lemma_map[word] = lemma
    except Exception as e:
        print(f"Error processing grammar productions: {e}")
        return sentence_tokens # Return original tokens on error

    if not word_to_lemma_map:
        print("Warning: No lexical rules found in the grammar to build a map.")
        # Fall through to map_tokens_to_lemmas, which will just return original tokens

    # 2. Map input tokens using the created map
    mapped_output = []
    for token in sentence_tokens:
        lemma = word_to_lemma_map.get(token.lower(), token)
        mapped_output.append(lemma)

    return mapped_output

In [10]:
with open(os.path.join(path_lexicons, "negation/negation.csv")) as f:
    negation_lexicon = pd.read_csv(f)

with open(os.path.join(path_lexicons, "uncertainty/uncertainty.csv")) as f:
    uncertainty_lexicon = pd.read_csv(f)

In [11]:
def process_document(doc_id, document_texts):
    """
    Process a document to detect negation and uncertainty
    
    Parameters:
        doc_id (str): Document ID
        document_texts (dict): Dictionary mapping line numbers to text
        
    Returns:
        list: List of prediction dictionaries
    """
    predictions = []
    
    for line_num, text in document_texts.items():
        if not text:
            continue
            
        processed_text = preprocess_text(text) # Preprocess text
        processed_text = tokenize_text(processed_text) # Tokenize text
        processed_text = parse_tokens_with_lexical_grammar(grammar, processed_text) # Lemmatize text
        
        # Find negation cues
        neg_cues = find_cues(processed_text, negation_lexicon)
        
        for neg_cue in neg_cues: # Process each negation cue
            predictions.append({
                "doc_id": doc_id,
                "line_number": f"{doc_id}_{line_num}",
                "result_id": f"neg_{len(predictions)}",
                "start": neg_cue["start"],
                "end": neg_cue["end"],
                "label": "NEG",
                "text": processed_text[neg_cue["start"]:neg_cue["end"]]
            }) # Add NEG prediction
            
            if neg_cue["lemma"] == "_no" or neg_cue["lemma"] == "_sin":
                scope = {}

                # define scope as everything after the cue
                scope["start"] = neg_cue["end"]
                scope["end"] = len(processed_text)
                scope["text"] = processed_text[scope["start"]:scope["end"]]
            
                if scope["start"] < scope["end"]: # Add NSCO prediction if scope is non-empty
                    predictions.append({
                        "doc_id": doc_id,
                        "line_number": f"{doc_id}_{line_num}",
                        "result_id": f"nsco_{len(predictions)}",
                        "start": scope["start"],
                        "end": scope["end"],
                        "label": "NSCO",
                        "text": scope["text"]
                    })
        
        # Find uncertainty cues
        unc_cues = find_cues(processed_text, uncertainty_lexicon)
        
        # Process each uncertainty cue
        for unc_cue in unc_cues:
            # Add UNC prediction
            predictions.append({
                "doc_id": doc_id,
                "line_number": f"{doc_id}_{line_num}",
                "result_id": f"unc_{len(predictions)}",
                "start": unc_cue["start"],
                "end": unc_cue["end"],
                "label": "UNC",
                "text": processed_text[unc_cue["start"]:unc_cue["end"]]
            })
            """
            
            scope = determine_scope(processed_text, unc_cue)
            
            # Add USCO prediction if scope is non-empty
            if scope["start"] < scope["end"]:
                predictions.append({
                    "doc_id": doc_id,
                    "line_number": line_num,
                    "result_id": f"usco_{len(predictions)}",
                    "start": scope["start"],
                    "end": scope["end"],
                    "label": "USCO",
                    "text": scope["text"]
                })
            """
    
    return predictions

**Text Extraction Process**

The document text extraction code works as follows:

- Create an empty dictionary `doc_texts` to store all document texts
- For each unique document ID in our dataframe:
   - Filter the dataframe to get only rows for this document
   - Create a dictionary `texts` to store lines for this document
   - For each unique line number in this document:
     - Filter to get only annotations for this line
     - Check each annotation in this line
     - This gives us the most complete text for this line
   - Store the line texts dictionary in our main document dictionary

This extraction preserves the document and line structure while giving us just the text content to work with

In [12]:
documents_directory = "../data/documents"

# loop over the documents
all_predictions = []

re_doc = re.compile(r"(\d+)\.txt$")

for document_name in tqdm(os.listdir(documents_directory)):
    match_obj = re_doc.match(document_name)
    if match_obj:
        doc_id = match_obj.group(1)

        # Load the document
        with open(os.path.join(documents_directory, f"{doc_id}.txt"), "r") as file:
            document_texts = {i: line.strip() for i, line in enumerate(file.readlines())}
        
        pred = process_document(doc_id, document_texts)
        all_predictions.extend(pred)

100%|██████████| 309/309 [00:02<00:00, 134.54it/s]


In [13]:
df_train_pred = pd.DataFrame(all_predictions)
print(f"Generated {len(df_train_pred)} predictions")

print("Prediction distribution:")
print(df_train_pred["label"].value_counts())

# Save predictions to file
output_file = os.path.join(path_predictions, "df_train_predictions.pkl")
pickle.dump(df_train_pred, open(output_file, "wb"))
print(f"Saved predictions to {output_file}")

Generated 10585 predictions
Prediction distribution:
label
NEG     5629
NSCO    4248
UNC      708
Name: count, dtype: int64
Saved predictions to ../data/predictions/df_train_predictions.pkl


In [14]:
df_train_pred

,doc_id,line_number,result_id,start,end,label,text
0,18806985,18806985_0,neg_0,168,169,NEG,[_sin]
1,18806985,18806985_0,nsco_1,169,184,NSCO,"[alergias, medicamentosas, calendario, vacunal..."
2,18806985,18806985_0,neg_2,180,181,NEG,[_no]
3,18806985,18806985_0,nsco_3,181,184,NSCO,"[requirio, ingreso, .]"
4,18806985,18806985_1,neg_4,0,1,NEG,[_no]
...,...,...,...,...,...,...,...
10580,18977663,18977663_34,unc_38,6,7,UNC,[_dudoso]
10581,18977663,18977663_38,neg_39,12,13,NEG,[_asintomatico]
10582,18977663,18977663_38,neg_40,14,15,NEG,[_sin]
10583,18977663,18977663_38,nsco_41,15,18,NSCO,"[mas, incidencias, .]"


In [15]:
df_train_filtered = df_train[(df_train["label"] == "NEG") | (df_train["label"] == "UNC")]
df_train_pred_filtered = df_train_pred[(df_train_pred["label"] == "NEG") | (df_train_pred["label"] == "UNC")]

In [16]:
print(len(df_train_filtered), len(df_train_pred_filtered))

4729 6337


**Accuracy Calculation**

In [17]:
import sys
sys.path.append("..")

from utils.metrics import calculate_entity_accuracy

All labels:

In [18]:
accuracy = calculate_entity_accuracy(df_train, df_train_pred, verbose=1)
print(f"Accuracy: {100*accuracy:.2f}%")

True Positives (TP): 6579
False Positives (FP): 2233
False Negatives (FN): 1024
Accuracy: 66.89%


Just NEG and UNC:

In [19]:
accuracy_filtered = calculate_entity_accuracy(df_train_filtered, df_train_pred_filtered, verbose=1)
print(f"Filtered accuracy: {100*accuracy_filtered:.2f}%")

True Positives (TP): 3792
False Positives (FP): 1416
False Negatives (FN): 99
Filtered accuracy: 71.45%
